In [27]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd

from scipy.optimize import curve_fit
from scipy.signal import find_peaks

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
qd.start_client('192.168.0.100')

QICK library version mismatch: 0.2.324 remote (the board), 0.2.302 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


In [29]:
default_config = qd.NVConfiguration()
default_config.mw_channel = 0
default_config.mw_nqz = 1
default_config.mw_gain = 5000


In [20]:
from qickdawg.arqick.arqick_cpmgxy8_endphase_sweep_200ps import CPMGXY8EndPhaseFineRes
from copy import copy

soc = qd.soc
config = copy(default_config)
config.mw_gain = 15000
config.freq_fMHz = 500
config.delay_tdds = 10000#193625
config.mw_pi2_tdds = 1000
config.n_cpmg = 1 # number of cpmg xy8 rounds
# remember the sweep is inclusive of the start and end values CHANGE
phase_low = 0
phase_high = 360
phase_step = 45
config.add_linear_sweep("end_phase", "pdegrees", phase_low, phase_high, delta = phase_step)
config.pulse_seq_delay_tus = 1
config.reps=1
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.inherent_trigger_to_pulses_delay_tns = 209.27
config.pmod_out_trig_delay_tus = 0
prog = CPMGXY8EndPhaseFineRes(config)
#prog.run_rounds(soc, rounds=0, start_src="external")
prog.run_rounds(soc, rounds=1)

100%|██████████| 9/9 [00:00<00:00, 9004.95it/s]


# CPMGXY8 with coarse resolution to check

In [42]:
from qickdawg.arqick.legacy.arqick_cpmg_XY8 import CPMGXY8
from copy import copy
soc = qd.soc
config = copy(default_config)
config.mw_gain = 15000
config.mw_pi2_tns = 50
config.freq_fMHz = 500 # in Hz
config.add_linear_sweep(name = "delay", unit = "tns", start = 10, stop = 100, delta=90)
config.n_cpmg = 1 # number of cpmg xy8 rounds
config.pulse_seq_delay_tus = 1
config.reps=1
config.pmod_out_pin = 0
config.pmod_out_pulse_width_tns = 100
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 209.27

prog = CPMGXY8(config)
prog.run_rounds(soc, rounds=0, start_src='external')


Requested 10 to 100 by 90
Instead using 9.765625 to 100.91145833333334 by 91.14583333333334 in 2 steps


# 200ps resolution now

In [7]:
import copy
from qickdawg.arqick.arqick_cpmgxy8_200ps import CPMGXY8FineRes
soc = qd.soc
config = copy.copy(default_config)

# MW params
config.mw_channel = 0
config.freq_fMHz = 500 # 200
config.mw_gain = 15000 # check user if do two amps for min max
config.mw_nqz = 1
# Timing params
config.mw_pi2_tdds = 50 # check user for min and max
# Sweep params
config.n_cpmg = 1 # check user for min and max
config.reps = 1
config.pulse_seq_delay_tus = 1
# should check the user to make sure min samples is something and max
config.add_unitless_linear_sweep("delay_tdds", 500, 510, delta=5) # the sweep is inclusive of the start and end values
# Triggering
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 0
prog = CPMGXY8FineRes(config)
prog.run_rounds(soc, rounds=1)

100%|██████████| 3/3 [00:00<00:00, 3005.95it/s]


# 200 ps rabi

In [74]:
import copy
from qickdawg.arqick.arqick_rabi_200ps import RabiFineRes
soc = qd.soc
config = copy.copy(default_config)

# MW params
config.mw_channel = 0
config.freq_fMHz = 500 # 200
config.mw_gain = 32000 # check user if do two amps for min max
config.mw_nqz = 1

# Sweep params
config.add_unitless_linear_sweep("mw_duration_tdds", 50, 55, delta=5) # the sweep is inclusive of the start and end values
config.reps = 1
config.pulse_seq_delay_tus = 1
# should check the user to make sure min samples is something and max
# Triggering
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns =0
prog = RabiFineRes(config)
prog.run_rounds(soc, rounds=1, start_src='external')

100%|██████████| 2/2 [00:04<00:00,  2.33s/it]


In [90]:
for i in range(2):
    print(i)

0
1


# Simple Pulse at 200ps

In [15]:
import copy
from qickdawg.nvpulsing.rfpulse_200ps import MWPulseFineRes
soc = qd.soc
config = copy.copy(default_config)

# MW params
config.mw_channel = 0
config.freq_fMHz = 500 # 200
config.mw_gain = 32000 # check user if do two amps for min max
config.mw_nqz = 1
config.mode_config = "oneshot" # "oneshot" or "periodic"
config.mw_duration_tdds = 48
# Sweep params
config.reps = 1
# should check the user to make sure min samples is something and max
# Triggering
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 209.27
prog = MWPulseFineRes(config)
prog.run_rounds(soc, rounds=1)

100%|██████████| 1/1 [00:00<00:00, 988.76it/s]


# Phase check

In [6]:
import copy
from qickdawg.nvpulsing.testIQ_200ps import MWPulseIQtest
soc = qd.soc
config = copy.copy(default_config)

# MW params
config.mw_channel = 0
config.freq_fMHz = 500 # 200
config.mw_gain = 32000 # check user if do two amps for min max
config.mw_nqz = 1
config.mw_duration_tdds = 48*3
config.btwn_mw_delay_tdds = 48*3
# Sweep params
config.reps = 1
# should check the user to make sure min samples is something and max
# Triggering
config.pmod_out_pin = 0 
config.pmod_out_pulse_width_tns = 50
config.pmod_out_trig_delay_tns = 0 #5000-198
config.inherent_trigger_to_pulses_delay_tns = 209.27
prog = MWPulseIQtest(config)
prog.run_rounds(soc, rounds=1)

100%|██████████| 1/1 [00:00<?, ?it/s]
